In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",      # Llama-3.1 2x faster
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-405B-bnb-4bit",    # 4bit for 405b!
    "unsloth/Mistral-Small-Instruct-2409",     # Mistral 22b 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!

    "unsloth/Llama-3.2-1B-bnb-4bit",           # NEW! Llama 3.2 models
    "unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
    "unsloth/Llama-3.2-3B-bnb-4bit",
    "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",

    "unsloth/Llama-3.3-70B-Instruct-bnb-4bit" # NEW! Llama 3.3 70B!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct", # or choose "unsloth/Llama-3.2-1B-Instruct"
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts, }

from datasets import load_dataset
dataset = load_dataset("mlabonne/FineTome-100k", split = "train")

from unsloth.chat_templates import standardize_sharegpt
dataset = standardize_sharegpt(dataset)
dataset = dataset.map(formatting_prompts_func, batched = True,)

In [ ]:
# Das hier ist das eigene finetuning per Prompts

# =========================
# INSERT THIS ENTIRE BLOCK BEFORE YOU CREATE THE SFTTrainer
# (i.e., after: model = FastLanguageModel.get_peft_model(...))
# =========================

from datasets import Dataset

# 1) Build a small example training dataset (edit/add your own examples)
train_convos = [
    {
        "conversations": [
            {"role": "user", "content": "Was ist die Wurzel von 3? Antworte kurz."},
            {"role": "assistant", "content": "√3 ≈ 1,732."},
        ]
    },
    {
        "conversations": [
            {"role": "user", "content": "Erkläre den Unterschied zwischen SFT und RLHF in 2 Sätzen."},
            {"role": "assistant", "content": "SFT ist überwachtes Lernen auf Beispieldialogen, bei dem das Modell Antworten imitiert. RLHF optimiert Antworten über ein Reward-/Präferenzsignal und formt Verhalten stärker in Richtung menschlicher Vorlieben."},
        ]
    },
    {
        "conversations": [
            {"role": "user", "content": "Gib mir eine kurze, strukturierte To-do-Liste für eine Mathe-Klausur."},
            {"role": "assistant", "content": "1) Themenliste erstellen. 2) Formeln sammeln. 3) Täglich Aufgaben rechnen. 4) Fehlerlog führen. 5) Probeklausuren unter Zeitdruck."},
        ]
    },
]

train_ds_raw = Dataset.from_list(train_convos)

# 2) Convert structured conversations -> single text string using the Llama chat template
def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(
            convo,
            tokenize=False,
            add_generation_prompt=False,
        )
        for convo in convos
    ]
    return {"text": texts}

train_ds = train_ds_raw.map(
    formatting_prompts_func,
    batched=True,
    remove_columns=["conversations"],
)

# 3) Quick sanity check (optional)
print("Train samples:", len(train_ds))
print(train_ds[0]["text"][:400])

In [ ]:
# Neu Initialisierung der Lora Gewichte

# 1) Basismodell neu laden (damit sind alle bisherigen LoRA-Änderungen weg)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

# 2) LoRA neu initialisieren (frische Adapter)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

# Set the custom fine-tuning label AFTER LoRA re-initialization
model._finetune_label = "Custom mini-dataset (German, concise style)"

In [ ]:
# Trainer FineTome Dataset

from trl import SFTConfig, SFTTrainer
from transformers import DataCollatorForSeq2Seq
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    packing = False, # Can make training 5x faster for short sequences.
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 60,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

In [ ]:
# Trainer eigenes Dataset

from trl import SFTConfig, SFTTrainer
from transformers import DataCollatorForSeq2Seq
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds, # Changed from 'dataset' to 'train_ds'
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    packing = False, # Can make training 5x faster for short sequences.
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 60,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)

In [ ]:
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

messages = [
    {"role": "user", "content": "Wie geht es?"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 128,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

In [ ]:
# =========================
# Simple CLI / Output Chat (no Gradio) — append at end of your Colab
# =========================

import torch
from unsloth import FastLanguageModel
from transformers import TextIteratorStreamer
from threading import Thread

FastLanguageModel.for_inference(model)  # fast inference

# Optional: set a system prompt (or keep None)
SYSTEM_PROMPT = None
# SYSTEM_PROMPT = "Du bist ein hilfreicher Assistent. Antworte knapp und korrekt."

GEN_KWARGS = dict(
    max_new_tokens=256,
    use_cache=True,
    temperature=0.8,
    min_p=0.1,
)

def generate_reply(messages, **gen_kwargs) -> str:
    """
    messages: list of dicts: {"role": "user"/"assistant"/"system", "content": "..."}
    Returns full assistant reply (string). Also streams tokens to output while generating.
    """
    # Build model inputs WITH attention_mask to avoid the warning
    enc = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )
    enc = {k: v.to("cuda") for k, v in enc.items()}

    streamer = TextIteratorStreamer(
        tokenizer,
        skip_prompt=True,
        skip_special_tokens=True,
    )

    # Run generation in a thread so we can stream tokens as they arrive
    t = Thread(
        target=model.generate,
        kwargs=dict(
            input_ids=enc["input_ids"],
            attention_mask=enc.get("attention_mask", None),
            streamer=streamer,
            **gen_kwargs,
        ),
    )
    t.start()

    # Stream to the notebook output and also accumulate final text
    out = []
    for token in streamer:
        print(token, end="", flush=True)
        out.append(token)
    print()  # newline at end

    return "".join(out).strip()

def chat_loop():
    """
    Interactive chat loop in notebook output.
    Commands:
      /exit  -> end
      /reset -> clear history
      /sys <text> -> set system prompt
    """
    global SYSTEM_PROMPT
    history = []

    print("Chat gestartet. Tippe deine Nachricht und drücke Enter.")
    print("Commands: /exit, /reset, /sys <text>")
    print("-" * 60)

    while True:
        try:
            user = input("Du: ").strip()
        except EOFError:
            break

        if not user:
            continue
        if user.lower() in ("/exit", "/quit"):
            print("Chat beendet.")
            break
        if user.lower() == "/reset":
            history = []
            print("Historie gelöscht.")
            continue
        if user.lower().startswith("/sys "):
            SYSTEM_PROMPT = user[5:].strip() or None
            print(f"System prompt gesetzt: {SYSTEM_PROMPT!r}")
            continue

        # Build messages from history
        messages = []
        if SYSTEM_PROMPT:
            messages.append({"role": "system", "content": SYSTEM_PROMPT})
        for u, a in history:
            messages.append({"role": "user", "content": u})
            messages.append({"role": "assistant", "content": a})
        messages.append({"role": "user", "content": user})

        print("Bot: ", end="", flush=True)
        reply = generate_reply(messages, **GEN_KWARGS)

        history.append((user, reply))

# Run the chat
chat_loop()

In [ ]:
model.save_pretrained("lora_model")  # Local saving
tokenizer.save_pretrained("lora_model")
# model.push_to_hub("your_name/lora_model", token = "...") # Online saving
# tokenizer.push_to_hub("your_name/lora_model", token = "...") # Online saving

In [ ]:
# =========================
# CONTROL: What is the model currently fine-tuned on?
# =========================

def print_finetune_status(model):
    print("\n=== Fine-Tuning Status ===")

    # 1) Check if PEFT / LoRA is active
    is_peft = hasattr(model, "peft_config") and model.peft_config is not None
    print(f"LoRA active: {is_peft}")

    if not is_peft:
        print("→ Model is BASE (no fine-tuning applied)")
        return

    # 2) List active adapters
    try:
        active_adapters = model.active_adapters
    except Exception:
        active_adapters = "unknown"

    print(f"Active adapter(s): {active_adapters}")

    # 3) Show LoRA configuration (rank etc.)
    for name, cfg in model.peft_config.items():
        print(f"\nAdapter name: {name}")
        print(f"  LoRA rank (r): {cfg.r}")
        print(f"  LoRA alpha:   {cfg.lora_alpha}")
        print(f"  Target mods:  {cfg.target_modules}")

    # 4) Optional: user-defined training label (if you set one)
    if hasattr(model, "_finetune_label"):
        print(f"\nUser training label: {model._finetune_label}")
    else:
        print("\nUser training label: NOT SET")
        print("→ Dataset origin cannot be inferred automatically.")

    print("==========================\n")


print_finetune_status(model)